In [2]:
import torch
import torchvision
import cv2
import matplotlib.pyplot as plt
import numpy as np
import math
from torchvision import transforms as T
%matplotlib inline

In [3]:
# Используем предобученную keypointrcnn_resnet50_fpn сеть для детекции поз
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = torchvision.models.detection.keypointrcnn_resnet50_fpn(pretrained=True)
model = model.to(device)  # Перемещение модели на GPU
model.eval()

c:\Users\Ghost\AppData\Local\Programs\Python\Python39\lib\site-packages\torchvision\models\_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
c:\Users\Ghost\AppData\Local\Programs\Python\Python39\lib\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=KeypointRCNN_ResNet50_FPN_Weights.COCO_V1`. You can also use `weights=KeypointRCNN_ResNet50_FPN_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


KeypointRCNN(
  (transform): GeneralizedRCNNTransform(
      Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
      Resize(min_size=(640, 672, 704, 736, 768, 800), max_size=1333, mode='bilinear')
  )
  (backbone): BackboneWithFPN(
    (body): IntermediateLayerGetter(
      (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
      (bn1): FrozenBatchNorm2d(64, eps=0.0)
      (relu): ReLU(inplace=True)
      (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
      (layer1): Sequential(
        (0): Bottleneck(
          (conv1): Conv2d(64, 64, kernel_size=(1, 1), stride=(1, 1), bias=False)
          (bn1): FrozenBatchNorm2d(64, eps=0.0)
          (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
          (bn2): FrozenBatchNorm2d(64, eps=0.0)
          (conv3): Conv2d(64, 256, kernel_size=(1, 1), stride=(1, 1), bias=False)
          (bn3): FrozenBatchNorm2d(256, eps=0.

In [4]:
# Создаем список опорных точек человека
keypoints = ['nose','left_eye','right_eye',\
'left_ear','right_ear','left_shoulder',\
'right_shoulder','left_elbow','right_elbow',\
'left_wrist','right_wrist','left_hip',\
'right_hip','left_knee', 'right_knee', \
'left_ankle','right_ankle']

In [40]:
def preprocess_video(video_path, target_size=(224, 224), target_fps=1, max_frames=100):
    """
    Предобработка видео: чтение, изменение размера и конвертация в RGB
    
    Args:
        video_path: путь к видеофайлу
        target_size: целевой размер кадра
        target_fps: целевая частота кадров
        max_frames: максимальное количество кадров для обработки
    
    Returns:
        frame_list: список предобработанных кадров
    """
    cap = cv2.VideoCapture(video_path) # загрузка видео 
    frameRate = cap.get(target_fps) # частота кадров
    frame_list = []
    
    while(cap.isOpened()):
        frameId = cap.get(1) # номер текущего кадра
        ret, frame = cap.read()
        if (ret != True or frameId >= max_frames):
            break
        elif (frameId % target_fps == 0):
            # Изменение размера
            frame_resized = cv2.resize(frame, target_size)
            # Конвертация BGR -> RGB
            frame_rgb = cv2.cvtColor(frame_resized, cv2.COLOR_BGR2RGB)
        
        frame_list.append(frame_rgb)
    cap.release()
    return frame_list

def draw_keypoints_per_person(img, all_keypoints, all_scores, confs, keypoint_threshold=2, conf_threshold=0.9):
    """
    Отрисовка ключевых точек на изображении
    
    Args:
        img: исходное изображение
        all_keypoints: все обнаруженные ключевые точки
        all_scores: оценки достоверности ключевых точек
        confs: уверенность детектора в обнаружении человека
        keypoint_threshold: порог для отрисовки ключевых точек
        conf_threshold: порог для отрисовки человека
    
    Returns:
        img_copy: изображение с нарисованными ключевыми точками
    """
    # Создаем спектр цветов для разных людей
    cmap = plt.get_cmap('rainbow')
    # Создаем копию изображения
    img_copy = img.copy()
    
    if len(all_keypoints) == 0:
        return img_copy
        
    color_id = np.arange(1,255, 255//len(all_keypoints)).tolist()[::-1]
    # Для каждого обнаруженного человека
    for person_id in range(len(all_keypoints)):
        # Проверяем степень уверенности детектора
        if confs[person_id] > conf_threshold:
            # Собираем опорные точки конкретного человека
            keypoints = all_keypoints[person_id, ...]
            # Собираем скоры для опорных точек
            scores = all_scores[person_id, ...]
            # Итерируем по каждой ключевой точке
            for kp in range(len(scores)):
                # Проверяем степень уверенности детектора опорной точки
                if scores[kp] > keypoint_threshold:
                    # Конвертируем массив опорных точек в список целых чисел
                    keypoint = tuple(map(int, keypoints[kp, :2].detach().cpu().numpy().tolist()))
                    # Выбираем цвет для данного человека
                    color = tuple(np.asarray(cmap(color_id[person_id])[:-1])*255)
                    # Рисуем кружок радиуса 5 вокруг точки
                    cv2.circle(img_copy, keypoint, 5, color, -1)

    return img_copy

def get_limbs_from_keypoints(keypoints):
    """
    Определение соединений между ключевыми точками (кости скелета)
    
    Args:
        keypoints: список названий ключевых точек
    
    Returns:
        limbs: список соединений между точками
    """
    limbs = [       
        [keypoints.index('right_eye'), keypoints.index('nose')],
        [keypoints.index('right_eye'), keypoints.index('right_ear')],
        [keypoints.index('left_eye'), keypoints.index('nose')],
        [keypoints.index('left_eye'), keypoints.index('left_ear')],
        [keypoints.index('right_shoulder'), keypoints.index('right_elbow')],
        [keypoints.index('right_elbow'), keypoints.index('right_wrist')],
        [keypoints.index('left_shoulder'), keypoints.index('left_elbow')],
        [keypoints.index('left_elbow'), keypoints.index('left_wrist')],
        [keypoints.index('right_hip'), keypoints.index('right_knee')],
        [keypoints.index('right_knee'), keypoints.index('right_ankle')],
        [keypoints.index('left_hip'), keypoints.index('left_knee')],
        [keypoints.index('left_knee'), keypoints.index('left_ankle')],
        [keypoints.index('right_shoulder'), keypoints.index('left_shoulder')],
        [keypoints.index('right_hip'), keypoints.index('left_hip')],
        [keypoints.index('right_shoulder'), keypoints.index('right_hip')],
        [keypoints.index('left_shoulder'), keypoints.index('left_hip')]
    ]
    return limbs

def draw_skeleton_per_person(img, all_keypoints, all_scores, confs, keypoint_threshold=2, conf_threshold=0.9):
    """
    Отрисовка скелета человека на изображении
    
    Args:
        img: исходное изображение
        all_keypoints: все обнаруженные ключевые точки
        all_scores: оценки достоверности ключевых точек
        confs: уверенность детектора в обнаружении человека
        keypoint_threshold: порог для отрисовки соединений
        conf_threshold: порог для отрисовки человека
    
    Returns:
        img_copy: изображение с нарисованным скелетом
    """
    cmap = plt.get_cmap('rainbow')
    img_copy = img.copy()
    limbs = get_limbs_from_keypoints(keypoints)  # Получаем соединения между точками
    
    if len(all_keypoints) == 0:
        return img_copy
        
    colors = np.arange(1, 255, 255//len(all_keypoints)).tolist()[::-1]
    for person_id in range(len(all_keypoints)):
        if confs[person_id] > conf_threshold:
            keypoints_person = all_keypoints[person_id, ...]

            for limb_id in range(len(limbs)):
                # Получаем координаты точек соединения
                limb_loc1 = keypoints_person[limbs[limb_id][0], :2].cpu().detach().numpy().astype(np.int32)
                limb_loc2 = keypoints_person[limbs[limb_id][1], :2].cpu().detach().numpy().astype(np.int32)
                limb_score = min(all_scores[person_id, limbs[limb_id][0]], all_scores[person_id, limbs[limb_id][1]])
                
                # Рисуем линию если уверенность достаточна
                if limb_score > keypoint_threshold:
                    color = tuple(np.asarray(cmap(colors[person_id])[:-1]) * 255)
                    cv2.line(img_copy, tuple(limb_loc1), tuple(limb_loc2), color, 5)

    return img_copy

def person_keypoints(all_keypoints, all_scores, confs, conf_threshold=0.9):
    """
    Извлечение ключевых точек наиболее уверенно обнаруженного человека
    
    Args:
        all_keypoints: все обнаруженные ключевые точки
        all_scores: оценки достоверности ключевых точек
        confs: уверенность детектора в обнаружении человека
        conf_threshold: порог для выбора человека
    
    Returns:
        Keypoints: ключевые точки выбранного человека
        Scores: оценки достоверности точек
    """
    Keypoints = None
    Scores = None
    if len(all_keypoints) == 0:
        return Keypoints, Scores
        
    for person_id in range(len(all_keypoints)):
        if confs[person_id] > conf_threshold:
            Keypoints = all_keypoints[person_id, ...].detach()[:,:-1]
            Scores = all_scores[person_id, ...].cpu().detach().numpy()
            break  # Берем только первого человека с высокой уверенностью
    return Keypoints, Scores

def cosine_distance(pose1, pose2):
    """
    Вычисление косинусного расстояния между двумя позами
    
    Args:
        pose1: первая поза (набор векторов)
        pose2: вторая поза (набор векторов)
    
    Returns:
        dist: вектор косинусных расстояний для каждой точки
    """
    if pose1 is None or pose2 is None:
        return np.array([1.0])  # Максимальное расстояние если позы не обнаружены
    
    lst = []
    for i in range(len(pose1)):
        # Проверка на нулевые векторы
        norm1 = np.linalg.norm(pose1[i])
        norm2 = np.linalg.norm(pose2[i])
        
        if norm1 == 0 or norm2 == 0:
            lst.append(0)  # Минимальное сходство если один из векторов нулевой
        else:
            cossin = pose1[i].dot(pose2[i]) / (norm1 * norm2)
            lst.append(cossin)
    
    dist = np.array(lst).reshape(-1,1)
    return 1 - dist  # Преобразуем сходство в расстояние

def weight_distance(pose1, pose2, conf1):
    """
    Вычисление взвешенного расстояния между позами с учетом достоверности
    
    Args:
        pose1: первая поза (плоский вектор)
        pose2: вторая поза (плоский вектор)
        conf1: веса достоверности для точек
    
    Returns:
        weighted_dist: взвешенное евклидово расстояние
    """
    if pose1 is None or pose2 is None or conf1 is None:
        return 1.0  # Максимальное расстояние если позы не обнаружены

    # Вычисляем взвешенное расстояние
    sum1 = 1.0 / np.sum(conf1) if np.sum(conf1) > 0 else 1.0
    sum2 = 0.0

    for i in range(len(pose1)):
        # Каждый индекс i имеет x и y, у которых одинаковая оценка достоверности
        conf_ind = i // 2  # Целочисленное деление для пар координат
        if conf_ind < len(conf1):
            sum2 += conf1[conf_ind] * np.linalg.norm(pose1[i] - pose2[i])

    weighted_dist = sum1 * sum2
    return weighted_dist

def Frame_comparison(Example, imitation):
    """
    Сравнение двух поз между примером и имитацией
    
    Args:
        Example: словарь с данными примера (ключевые точки, scores, confidences)
        imitation: словарь с данными имитации
    
    Returns:
        dist: косинусное расстояние
        weighted_dist: взвешенное расстояние
    """
    # Извлекаем ключевые точки для примера и имитации
    model_key_points, Y_scores = person_keypoints(Example["keypoints"], Example["keypoints_scores"], Example["scores"])
    input_key_points, X_scores = person_keypoints(imitation["keypoints"], imitation["keypoints_scores"], imitation["scores"])
    
    # Проверка на наличие обнаруженных поз
    if model_key_points is None or input_key_points is None:
        return np.array([1.0]), 1.0
 
    model_key_points = model_key_points.cpu().numpy()
    input_key_points = input_key_points.cpu().numpy()
 
    # Функции для добавления и удаления столбца единиц (для аффинного преобразования)
    pad = lambda x: np.hstack((x, np.ones((x.shape[0], 1))))
    unpad = lambda x: x[:,:-1]
 
    Y = pad(model_key_points)
    X = pad(input_key_points)
    
    # Вычисление аффинного преобразования для выравнивания поз
    A, res, rank, s = np.linalg.lstsq(X, Y, rcond=None)
    A[np.abs(A) < 1e-10] = 0

    transform = lambda x: unpad(np.dot(pad(x), A))
    input_transform = transform(input_key_points)
    
    X = input_transform
    Y = unpad(Y)
    
    pose1 = Y.flatten()  # Преобразуем в 1D для weight_distance
    pose2 = X.flatten()
    
    # Вычисляем оба типа расстояний
    dist = cosine_distance(Y, X)  # Используем 2D для cosine_distance
    weighted_dist = weight_distance(pose1, pose2, X_scores)
    
    return dist, weighted_dist

def create_results_video(example_frames, imitation_frames, 
                               example_skeletons, imitation_skeletons,
                               dist_list, weighted_list, output_path, fps=10):
    """
    Создание видео с результатами сравнения поз
    
    Args:
        example_frames: кадры примера
        imitation_frames: кадры имитации  
        example_skeletons: скелеты примера
        imitation_skeletons: скелеты имитации
        dist_list: список косинусных расстояний
        weighted_list: список взвешенных расстояний
        output_path: путь для сохранения видео
        fps: частота кадров видео
    """
    min_len = min(len(example_frames), len(imitation_frames), 
                 len(example_skeletons), len(imitation_skeletons),
                 len(dist_list), len(weighted_list))
    
    # Компактные размеры для отображения на экране
    height, width = 180, 240
    combined_width = width * 2
    combined_height = height + 200  # Место для аналитики
    
    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    out = cv2.VideoWriter(output_path, fourcc, fps, (combined_width, combined_height))
    
    print("Создание видео с результатами...")
    
    for i in range(min_len):
        # Создаем белый фон
        combined_frame = np.ones((combined_height, combined_width, 3), dtype=np.uint8) * 255
        
        # Уменьшаем и размещаем кадры скелетонов
        example_small = cv2.resize(example_skeletons[i], (width, height))
        imitation_small = cv2.resize(imitation_skeletons[i], (width, height))
        
        example_bgr = cv2.cvtColor(example_small, cv2.COLOR_RGB2BGR)
        imitation_bgr = cv2.cvtColor(imitation_small, cv2.COLOR_RGB2BGR)
        
        combined_frame[0:height, :width] = example_bgr
        combined_frame[0:height, width:width*2] = imitation_bgr
        
        # Добавляем заголовки
        cv2.putText(combined_frame, "EXAMPLE", (10, height + 20), 
                   cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 0, 0), 2)
        cv2.putText(combined_frame, "IMITATION", (width + 10, height + 20), 
                   cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 0, 0), 2)
        
        # ОСНОВНЫЕ МЕТРИКИ
        metrics_y = height + 45
        
        # Текущие значения метрик
        cv2.putText(combined_frame, f"Frame: {i+1}/{min_len}", (20, metrics_y), 
                   cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 0, 0), 1)
        cv2.putText(combined_frame, f"Cosine: {dist_list[i]:.4f}", (20, metrics_y + 20), 
                   cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 0, 150), 1)
        cv2.putText(combined_frame, f"Weighted: {weighted_list[i]:.1f}", (20, metrics_y + 40), 
                   cv2.FONT_HERSHEY_SIMPLEX, 0.5, (150, 0, 0), 1)
        
        # Средние значения по всем кадрам
        avg_cosine = np.mean(dist_list[:i+1])
        avg_weighted = np.mean(weighted_list[:i+1])
        
        cv2.putText(combined_frame, f"Avg Cosine: {avg_cosine:.4f}", (width + 20, metrics_y), 
                   cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 100, 0), 1)
        cv2.putText(combined_frame, f"Avg Weighted: {avg_weighted:.1f}", (width + 20, metrics_y + 20), 
                   cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 100, 0), 1)
        
        #ГРАФИК КОСИНУСНОЙ ДИСТАНЦИИ
        graph_y = metrics_y + 60
        graph_width = 400
        graph_height = 40
        
        # Рисуем рамку графика
        cv2.rectangle(combined_frame, (20, graph_y), 
                     (20 + graph_width, graph_y + graph_height), (200, 200, 200), 1)
        
        # Горизонтальная линия для ориентира (порог качества 0.005)
        guide_y = graph_y + graph_height - int(0.005 * graph_height / 0.01)
        cv2.line(combined_frame, (20, guide_y), (20 + graph_width, guide_y), 
                (200, 200, 0), 1)
        cv2.putText(combined_frame, "0.005", (25, guide_y - 2), 
                   cv2.FONT_HERSHEY_SIMPLEX, 0.3, (100, 100, 0), 1)
        
        # Рисуем график косинусной дистанции
        if i > 0:
            for j in range(1, i+1):
                if j < len(dist_list):
                    x1 = 20 + int((j-1) * graph_width / min_len)
                    y1 = graph_y + graph_height - int(dist_list[j-1] * graph_height / 0.01)
                    x2 = 20 + int(j * graph_width / min_len)
                    y2 = graph_y + graph_height - int(dist_list[j] * graph_height / 0.01)
                    
                    # Ограничиваем координаты в пределах графика
                    y1 = max(graph_y, min(graph_y + graph_height, y1))
                    y2 = max(graph_y, min(graph_y + graph_height, y2))
                    
                    cv2.line(combined_frame, (x1, y1), (x2, y2), (0, 0, 255), 1)
        
        # Текущая точка на графике
        if i < len(dist_list):
            current_x = 20 + int(i * graph_width / min_len)
            current_y_pos = graph_y + graph_height - int(dist_list[i] * graph_height / 0.01)
            current_y_pos = max(graph_y, min(graph_y + graph_height, current_y_pos))
            cv2.circle(combined_frame, (current_x, current_y_pos), 3, (0, 0, 200), -1)
        
        # ОЦЕНКА КАЧЕСТВА ВЫПОЛНЕНИЯ
        quality_y = graph_y + graph_height + 20
        
        # Определяем качество на основе средней косинусной дистанции
        if avg_cosine < 0.005:
            quality = "EXCELLENT"
            color = (0, 200, 0)  # Зеленый
        elif avg_cosine < 0.01:
            quality = "VERY GOOD" 
            color = (0, 150, 150)  # Голубой
        elif avg_cosine < 0.03:
            quality = "GOOD"
            color = (0, 100, 255)  # Оранжевый
        else:
            quality = "NEEDS PRACTICE"
            color = (0, 0, 255)  # Красный
        
        cv2.putText(combined_frame, f"QUALITY: {quality}", 
                   (width + 20, quality_y), cv2.FONT_HERSHEY_SIMPLEX, 0.6, color, 2)
        
        # Прогресс выполнения анализа
        progress_width = 150
        progress = (i + 1) / min_len
        cv2.rectangle(combined_frame, (width + 20, quality_y + 10), 
                     (width + 20 + progress_width, quality_y + 20), (200, 200, 200), -1)
        cv2.rectangle(combined_frame, (width + 20, quality_y + 10), 
                     (width + 20 + int(progress_width * progress), quality_y + 20), 
                     (0, 150, 0), -1)
        cv2.putText(combined_frame, f"{progress*100:.0f}%", 
                   (width + 20 + progress_width + 5, quality_y + 32), 
                   cv2.FONT_HERSHEY_SIMPLEX, 0.4, (0, 0, 0), 1)
        
        out.write(combined_frame)
        
        if (i + 1) % 50 == 0:
            print(f"Обработано кадров: {i+1}/{min_len}")
    
    out.release()
    print(f"Видео с результатами сохранено: {output_path}")

def display_complete_analysis():
    """
    Комплексное отображение всех результатов анализа
    """
    print("=" * 60)
    print("РЕЗУЛЬТАТЫ АНАЛИЗА")
    print("=" * 60)
    
    # Показываем метрики
    print(f"\n📊 ОСНОВНЫЕ МЕТРИКИ:")
    print(f"   Средняя косинусная дистанция: {averege_dist:.6f}")
    print(f"   Средняя взвешенная дистанция: {averege_weighted_dist:.4f}")
    print(f"   Проанализировано кадров: {len(dist_list)}")
    
    # Оценка качества
    if averege_dist < 0.005:
        rating = "⭐️⭐️⭐️⭐️⭐️ ОТЛИЧНО"
        color = "🟢"
    elif averege_dist < 0.01:
        rating = "⭐️⭐️⭐️⭐️ ОЧЕНЬ ХОРОШО" 
        color = "🟡"
    elif averege_dist < 0.03:
        rating = "⭐️⭐️⭐️ ХОРОШО"
        color = "🟠"
    else:
        rating = "⭐️⭐️ НУЖНА ПРАКТИКА"
        color = "🔴"
    
    print(f"\n{color} ОЦЕНКА КАЧЕСТВА: {rating}")
    

# Преобразование для входных данных модели
transform = T.Compose([T.ToTensor()])


In [46]:
# Загрузка и предобработка видео
videoFile_Example = "Data/Reference.mp4"
videoFile_Imitation = "Data/Imitation.mp4"
video_Example = preprocess_video(videoFile_Example, max_frames=280)
video_Imitation = preprocess_video(videoFile_Imitation, max_frames=280)
print(f"Количество кадров примера: {len(video_Example)}")
print(f"Количество кадров имитации: {len(video_Imitation)}")

Количество кадров примера: 280
Количество кадров имитации: 231


In [47]:
# Обработка видео примера
Example_Modeled = []
print("Обработка примера...")
for i, frame in enumerate(video_Example):
    img_tensor = transform(frame).to(device)
    with torch.no_grad():
        result = model([img_tensor])[0]
    Example_Modeled.append(result)
    if (i + 1) % 50 == 0:
        print(f"Обработано кадров примера: {i+1}/{len(video_Example)}")
print("Обработка завершена")

Обработка примера...
Обработано кадров примера: 50/280
Обработано кадров примера: 100/280
Обработано кадров примера: 150/280
Обработано кадров примера: 200/280
Обработано кадров примера: 250/280
Обработка завершена


In [48]:
# Сравнение поз и создание визуализаций
print("Сравнение поз и создание визуализаций...")
Example_skeletal_frames = []
Imitation_skeletal_frames = []
Combined_skeletal_frames = []
dist_list = []
weighted_dist_list = []
for i, frame in enumerate(video_Imitation):
    img_tensor = transform(frame).to(device)
    with torch.no_grad():
        result = model([img_tensor])[0]
    
    # Сравнение поз текущего кадра
    dist, weighted_dist = Frame_comparison(Example_Modeled[i], result)
    dist_list.append(dist.mean())  # Усредняем по всем ключевым точкам
    weighted_dist_list.append(weighted_dist)
    
    # Создание визуализаций скелетонов
    Example_skeletal_frames.append(draw_skeleton_per_person(video_Example[i], Example_Modeled[i]["keypoints"], Example_Modeled[i]["keypoints_scores"], Example_Modeled[i]["scores"], keypoint_threshold=2))
    Imitation_skeletal_frames.append(draw_skeleton_per_person(video_Imitation[i], result["keypoints"], result["keypoints_scores"], result["scores"], keypoint_threshold=2))
    
    if (i + 1) % 50 == 0:
        print(f"Обработано кадров имитации: {i+1}/{len(video_Imitation)}")
print("Обработка завершена")
# Вычисление итоговых метрик
averege_dist = np.mean(dist_list) if dist_list else 1.0
averege_weighted_dist = np.mean(weighted_dist_list) if weighted_dist_list else 1.0
print(f"Средняя косинусная дистанция = {averege_dist:.4f}")
print(f"Средняя взвешенная дистанция = {averege_weighted_dist:.4f}")

Сравнение поз и создание визуализаций...
Обработано кадров имитации: 50/231
Обработано кадров имитации: 100/231
Обработано кадров имитации: 150/231
Обработано кадров имитации: 200/231
Обработка завершена
Средняя косинусная дистанция = 0.0032
Средняя взвешенная дистанция = 14.7756


In [49]:
# Создание видео с результатами
create_results_video(
    video_Example,
    video_Imitation, 
    Example_skeletal_frames,
    Imitation_skeletal_frames,
    dist_list,
    weighted_dist_list,
    "results/performance_analysis1.mp4",
    fps=10
)

print("Анализ завершен! Результаты сохранены в папке 'results'")
display_complete_analysis()

Создание видео с результатами...
Обработано кадров: 50/231
Обработано кадров: 100/231
Обработано кадров: 150/231
Обработано кадров: 200/231
Видео с результатами сохранено: results/performance_analysis1.mp4
Анализ завершен! Результаты сохранены в папке 'results'
РЕЗУЛЬТАТЫ АНАЛИЗА

📊 ОСНОВНЫЕ МЕТРИКИ:
   Средняя косинусная дистанция: 0.003204
   Средняя взвешенная дистанция: 14.7756
   Проанализировано кадров: 231

🟢 ОЦЕНКА КАЧЕСТВА: ⭐️⭐️⭐️⭐️⭐️ ОТЛИЧНО
